# E-INV-P0 — Inverse-Channel Pilot  **(v3.1)**

> **v3.1 fixes a real bug found by the S1 gate**: `wiener_dft` was inverted — it preserved
> the strong spectral peaks (JPEG blocking, CFA, ISP shading) that are *shared* by every
> device of a camera model and zeroed the flat noise floor where the device-unique PRNU
> lives. Synthetic check: the old filter pushed cross-device same-model NCC from 0.49 to
> **0.90**; the corrected one gives ~0.00 and raises device-unique retention 0.35 → 0.50.
> S1 now also reports the κ/device/noise variance split and gates on the same-model-pair
> AUC rather than the pooled figure.

**Passive device-fingerprint leakage into fine-tuned generative model weights.**
Single file, staged S0–S7, hard gates, resumable, fp32 measurement path, paired generation seeds.
Math + confound registry: `E_INV_P0_design.md` (§14 = v3 amendments).

### Novelty framing this notebook is built to defend
> First controlled test of whether a **naturally occurring, passive, device-specific** sensor
> fingerprint — no watermark, no injection — is internalized by a latent text-to-image model
> during ordinary LoRA adaptation and re-emitted in images generated **without image
> conditioning**, discriminating **two physical devices of the same camera model**.

Prior art bounding the claim: Yu et al. ICCV'21 (artificial fingerprints), ProMark CVPR'24,
CoprGuard CVPR'25 — all *injected* signals. Never claim "first to show fingerprints transfer".

### v3 changes (from the external survey + my own audit of its recommendations)
| # | Change | Origin |
|---|---|---|
| 1 | λ_min from an **empirical null** (off-peak of the mean cross-correlation surface), with (z_α+z_β) | survey 5.1 was right that v2 used the α-threshold (50% power); the empirical null supersedes its analytic fix, which still assumed pixel independence |
| 2 | **S1b: exact SD-3.5 VAE round-trip gate** — measure fingerprint survival, don't assume it from the density law | survey 5.4 |
| 3 | **S1c: causal arms** A_clean / A_swap, with **E1⊥E2** (clean with K̂^E1, measure with K̂^E2) | survey 5.3; the E1/E2 split is mine — cleaning and testing with the same K̂ manufactures a negative artifact |
| 4 | **Primary = intersection–union test** on the symmetric same-model contrast: S_A>0 ∧ S_B>0. No α adjustment (IUT is exact at level α) | survey 5.2/§6 substance; IUT framing is mine |
| 5 | **Paired generation seeds** — every adapter sees the identical (prompt, ε) bank | survey 5.7 |
| 6 | Copy **ensemble**: DINOv2 full + 4 quadrant crops, pHash, residual **peakedness** | survey 5.6, but residual-NN *max* would flag the leakage itself as copying — peakedness (top-1 vs median) is the correct discriminator |
| 7 | **Sign-agnostic** secondary: two-sided test + energy-distance permutation | survey 5.10 |
| 8 | **C0–C4 ladder** with a causal rung; **Holm** within the secondary family, hierarchically gated | survey §8; multiplicity plan is mine — the survey added ~9 statistics without one |
| 9 | Fixed-crop claim reworded (no "does not occur anywhere") | survey 5.9 |
| 10 | **All training crops materialized as PNG** for every arm | mine — raw arms would train on JPEG while clean/swap arms trained on PNG: a format confound that would have invalidated the causal comparison |

Declined: 50-prompt primary bank (pulls generations off the fine-tuned mode; kept as an optional
secondary bank), dropping device D (its cross-model null absorbs generation-process artifacts
that the same-model contrast cannot).

**Run order:** S0 → S1 → S1b → S1c → S2 → S3 → S4 → S5 → S6 → S7.

In [ ]:
# %% CONFIG — frozen pre-registration (protocol SHA excludes session fields)
import hashlib, json, os
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    # ---- session (NOT hashed) ----
    STAGES: tuple = ("S0",)
    DRIVE_ROOT: str = "/content/drive/MyDrive"
    DRESDEN_DIR: str = "forensic_datasets/dresden/Dresden_Exp"
    OUT_DIR: str = "inv_channel/E_INV_P0_v3"
    SKIP_DRIVE: bool = False
    LOCAL_CACHE: bool = True
    LOCAL_DIR: str = "/content/e_inv_cache"
    # ---- protocol (hashed) ----
    HF_MODEL: str = "stabilityai/stable-diffusion-3.5-medium"
    FALLBACK_512: bool = False           # native 512 centre crop, never a resize
    PREF_PAIR_MODEL: str = "Nikon_D200"
    DEV_C_PREF: str = "Agfa_DC-733s_0"
    DEV_D_PREF: str = "Nikon_D70_1"
    DEV_D2_PREF: str = "Agfa_DC-830i_0"
    CROP: int = 1024
    # splits: E1 (cleaning/injection fingerprint) | E2 (measurement fingerprint) | T | H
    N_E1_MAIN: int = 80                  # roles A,B only
    N_E2_MAIN: int = 140                 # roles A,B
    N_E2_AUX: int = 60                   # roles C,D,D2 (no E1)
    N_T: int = 50
    N_H: int = 40
    GUARD: int = 10
    SCENE_SIM_HALT: float = 0.95
    # forensics
    DENOISE_SIGMA0: float = 2.0
    WAVELET: str = "db8"; WAV_LEVELS: int = 4
    PCE_EXCL: int = 11
    CLEAN_ALPHA: float = 1.0             # strength of fingerprint suppression
    SUPPRESS_GATE: float = 5.0           # required rho reduction factor on held-out reals
    VAE_AUC_GATE: float = 0.75           # S1b: post-VAE same-vs-cross device AUC floor
    # LoRA
    LORA_RANK: int = 16
    LORA_TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    CEIL_RANK: int = 64
    CEIL_TARGETS: tuple = ("to_q","to_k","to_v","to_out.0","ff.net.0.proj","ff.net.2")
    LR: float = 1e-4; STEPS: int = 2000; CEIL_STEPS: int = 3000
    BATCH: int = 2; GRAD_ACC: int = 2
    CAPTION: str = "a photograph, sks style"
    TRAIN_SEEDS: tuple = (0,1,2)         # for the 3-seed arms
    # generation — PAIRED across every adapter
    G_PER_ADAPTER: int = 500
    GEN_SEED_BASE: int = 770000
    PROMPT_BANK: str = "uniform"         # "uniform" (primary) | "diverse" (secondary)
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    # statistics
    ALPHA: float = 0.01                  # primary IUT, one-sided
    ALPHA_SEC: float = 0.05              # secondary family, Holm-corrected
    POWER: float = 0.99
    N_PERM: int = 10000; N_BOOT: int = 10000
    COPY_DINO: float = 0.90; COPY_CROP: float = 0.92; COPY_PHASH: int = 8
    COPY_RESID_Z: float = 6.0            # residual peakedness z (top1 vs median), NOT raw max
    AUC_GATE: float = 0.99          # pooled, all roles
    AUC_PAIR_GATE: float = 0.95     # SAME-MODEL pair A vs B — the binding one
    KAPPA_CEIL: float = 0.10        # NCC(K_A,K_B) after cleaning
    DEVFRAC_FLOOR: float = 0.25     # device-unique share of the paired contrast
    LAMBDA_MIN_CEIL: float = 0.05

C = CFG()
MEAS = 512 if C.FALLBACK_512 else C.CROP
_SESSION = {"STAGES","DRIVE_ROOT","DRESDEN_DIR","OUT_DIR","SKIP_DRIVE","LOCAL_CACHE","LOCAL_DIR"}
CFG_SHA = hashlib.sha256(json.dumps({k:v for k,v in asdict(C).items() if k not in _SESSION},
                                    sort_keys=True, default=str).encode()).hexdigest()
ROLES = ("A","B","C","D","D2")

def stage(name):
    if name in C.STAGES: return True
    print(f"[{name}] SKIPPED — not in C.STAGES={C.STAGES}")
    return False

# ---- ARM REGISTRY: the single source of truth for training / generation / measurement ----
def all_arms():
    """(tag, role, variant, seed, cap). variant in {raw, clean, swap}. base has no adapter."""
    A = [("base", None, None, None, None)]
    for s in C.TRAIN_SEEDS:
        for r in ("A","B","D"):
            A.append((f"{r}_raw_s{s}_r16", r, "raw", s, "r16"))
    A.append(("D2_raw_s0_r16", "D2", "raw", 0, "r16"))     # secondary null
    A.append(("A_clean_s0_r16", "A", "clean", 0, "r16"))   # causal: fingerprint suppressed
    A.append(("A_swap_s0_r16",  "A", "swap",  0, "r16"))   # causal: K_A -> K_B
    A.append(("A_raw_s0_ceil",  "A", "raw",   0, "ceil"))  # capacity ceiling
    A.append(("D_raw_s0_ceil",  "D", "raw",   0, "ceil"))
    return A

def all_tags(): return [a[0] for a in all_arms()]
def arm_of(tag): return next(a for a in all_arms() if a[0]==tag)

print(f"[CONFIG] protocol SHA-256 = {CFG_SHA}")
print(f"[CONFIG] MEAS={MEAS}  arms={len(all_arms())}  stages={C.STAGES}")


In [ ]:
# %% ENV
import subprocess, sys, glob, re, math, time, shutil, itertools
import csv as _csv
def _pip(p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)

if not C.SKIP_DRIVE:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
ROOT = os.path.join(C.DRIVE_ROOT, C.OUT_DIR) if not C.SKIP_DRIVE else "/content/E_INV_P0_local"
for d in ["","manifest","fingerprints","adapters","gens","csv","figs","train_png","surfaces"]:
    os.makedirs(os.path.join(ROOT,d), exist_ok=True)
CACHE = C.LOCAL_DIR if C.LOCAL_CACHE else None
if CACHE: os.makedirs(CACHE, exist_ok=True)
print("[ENV] ROOT =", ROOT, "| CACHE =", CACHE)

_pip(["PyWavelets","diffusers>=0.31","transformers","peft","accelerate","safetensors",
      "sentencepiece","protobuf","timm","ImageHash"])
import numpy as np, pywt, torch
import torch.nn.functional as tF
from PIL import Image
import scipy.ndimage as ndi
Image.MAX_IMAGE_PIXELS = None
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("[ENV] device:", DEV, torch.cuda.get_device_name(0) if DEV=="cuda" else "")
np.seterr(divide="raise", invalid="raise")

if any(s in C.STAGES for s in ("S1b","S1c","S2","S3")):
    from huggingface_hub import login
    tok = os.environ.get("HF_TOKEN","")
    if tok: login(token=tok)
    elif any(s in C.STAGES for s in ("S1b","S2","S3")):
        print("[ENV] set os.environ['HF_TOKEN'] — SD-3.5 is a gated repo.")

def append_row(p, hdr, row):
    new = not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(hdr)+"\n")
        f.write(",".join(str(x) for x in row)+"\n")

def done_keys(p, cols):
    if not os.path.exists(p): return set()
    import csv as _c
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _c.DictReader(f)}

def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")


In [ ]:
# %% FORENSIC CORE — native crop (never resampled), fp32 residuals, K̂, PCE, clean/swap
def _cp(fp):
    return os.path.join(CACHE, hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")

def load_lum_crop(fp, use_cache=True):
    if CACHE and use_cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        if min(W,H)<MEAS: HALT(f"{fp}: {W}x{H} < MEAS {MEAS}")
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    lum=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if CACHE and use_cache: np.save(_cp(fp), lum)
    return lum

def load_rgb_crop(fp):
    """Native centre crop, RGB uint8-valued float. No resize, ever."""
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    return a

def _conv2(x,k):
    return tF.conv2d(torch.from_numpy(np.ascontiguousarray(x))[None,None],
                     torch.from_numpy(k)[None,None], padding=k.shape[0]//2)[0,0].numpy()

def wavelet_residual(img):
    s0=np.float32(C.DENOISE_SIGMA0)**2
    co=pywt.wavedec2(img, C.WAVELET, level=C.WAV_LEVELS, mode="periodization")
    out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        band=[]
        for c in (cH,cV,cD):
            c=c.astype(np.float32); vmin=None
            for w in (3,5,7,9):
                v=np.maximum(_conv2(c*c, np.ones((w,w),np.float32)/(w*w))-s0, 0.0)
                vmin=v if vmin is None else np.minimum(vmin,v)
            band.append(c*(s0/(vmin+s0)))
        out.append(tuple(band))
    n=pywt.waverec2(out, C.WAVELET, mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0.0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)

def zero_mean(K):
    K=K-K.mean(axis=1,keepdims=True); return K-K.mean(axis=0,keepdims=True)

def wiener_dft(K, sigma=None):
    """Suppress structured (periodic) spectral components, KEEP the flat noise floor.
    v3.0 had this inverted: it zeroed low-magnitude bins and preserved the strong peaks,
    i.e. it retained exactly the JPEG-blocking / CFA / ISP artefacts that are shared across
    every device of a camera model. Verified on synthetic data: old filter drove cross-device
    same-model NCC from 0.49 -> 0.90; this one drives it to ~0.00 while raising device-unique
    retention from 0.35 -> 0.50."""
    F=np.fft.fft2(K.astype(np.float32))
    mag=(np.abs(F)/K.shape[0]).astype(np.float32)
    if sigma is None: sigma=float(np.median(mag))/0.6745
    s2=np.float32(sigma**2); vmin=None
    for w in (3,5,7,9):
        k=np.ones((w,w),np.float32)/(w*w)
        mu=_conv2(mag,k); v=np.maximum(_conv2(mag*mag,k)-mu*mu,0.0)
        vmin=v if vmin is None else np.minimum(vmin,v)
    return np.real(np.fft.ifft2(F*(s2/(vmin+s2)).astype(np.float32))).astype(np.float32)

def clean_K(K): return wiener_dft(zero_mean(K))

def defect_split(K):
    med=ndi.median_filter(K,size=5); res=K-med
    thr=5.0*float(np.median(np.abs(res)))/0.6745; m=np.abs(res)>thr
    Kd=np.where(m,K,0.0).astype(np.float32)
    return Kd,(K-Kd).astype(np.float32),float(m.mean())

def estimate_K(files):
    num=den=None
    for fp in files:
        Y=load_lum_crop(fp); W=wavelet_residual(Y)
        num=W*Y if num is None else num+W*Y; den=Y*Y if den is None else den+Y*Y
    return clean_K((num/np.maximum(den,1e-6)).astype(np.float32))

def ncc0(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))

def xcorr_surface(a,b):
    """Normalized cross-correlation surface; C[0,0] == ncc0(a,b). All shifts at once."""
    a=a-a.mean(); b=b-b.mean()
    a=a/(np.linalg.norm(a)+1e-12); b=b/(np.linalg.norm(b)+1e-12)
    return np.real(np.fft.ifft2(np.fft.fft2(a)*np.conj(np.fft.fft2(b)))).astype(np.float32)

def offpeak_mask(shape):
    e=C.PCE_EXCL//2; m=np.ones(shape,bool)
    m[:e+1,:e+1]=m[:e+1,-e:]=m[-e:,:e+1]=m[-e:,-e:]=False
    return m

def pce_from_surface(Cs):
    m=offpeak_mask(Cs.shape); off=float((Cs[m]**2).mean())
    c0=float(Cs[0,0]); p0=math.copysign(c0*c0/max(off,1e-24), c0)
    i=np.unravel_index(np.argmax(np.abs(Cs)),Cs.shape); cm=float(Cs[i])
    return p0, math.copysign(cm*cm/max(off,1e-24), cm), (int(i[0]),int(i[1]))

def rho_pair(Wz,Z,K): return ncc0(Wz,Z*K), ncc0(Wz,K)

# ---- causal-arm image operators (design §14; applied with K̂ from E1, measured with K̂ from E2)
def apply_clean(rgb, K1, alpha=None):
    """Suppress device fingerprint: Y_clean = Y * (1 - a*K1). K1 luminance-derived, applied per channel."""
    a = C.CLEAN_ALPHA if alpha is None else alpha
    return np.clip(rgb*(1.0 - a*K1[...,None]), 0, 255).astype(np.float32)

def apply_swap(rgb, K1_from, K1_to, alpha=None):
    """Suppress K_from then inject K_to. Pipeline positive control (NOT a novelty claim —
    injected fingerprints are Yu et al./ProMark territory)."""
    a = C.CLEAN_ALPHA if alpha is None else alpha
    return np.clip(rgb*(1.0 - a*K1_from[...,None])*(1.0 + a*K1_to[...,None]), 0, 255).astype(np.float32)


In [ ]:
# %% S0 — MANIFEST: discovery, E1|E2|T|H guard-banded splits, scene audit
FNAME_RE = re.compile(r"^(?P<model>.+)_(?P<inst>\d+)_(?P<shot>\d+)\.(jpe?g|JPE?G|png|PNG)$")
def has_E1(role): return role in ("A","B")
def n_e1(role): return C.N_E1_MAIN if has_E1(role) else 0
def n_e2(role): return C.N_E2_MAIN if role in ("A","B") else C.N_E2_AUX
def need_for(role):
    n_splits = 4 if has_E1(role) else 3
    return n_e1(role)+n_e2(role)+C.N_T+C.N_H+(n_splits-1)*C.GUARD

def discover_devices():
    base=os.path.join(C.DRIVE_ROOT, C.DRESDEN_DIR)
    if not os.path.isdir(base): HALT(f"Dresden dir not found: {base}")
    dv={}
    for fp in glob.glob(os.path.join(base,"**","*.*"), recursive=True):
        m=FNAME_RE.match(os.path.basename(fp))
        if not m: continue
        dv.setdefault(f"{m['model']}_{m['inst']}",{"model":m["model"],"files":[]})["files"].append(fp)
    for v in dv.values(): v["files"].sort()
    return dv

def uniform_filter(files):
    from collections import Counter
    sizes=[]
    for fp in files:
        try:
            with Image.open(fp) as im: sizes.append((fp, im.size))
        except Exception: sizes.append((fp, None))
    cnt=Counter(s for _,s in sizes if s)
    if not cnt: return [], len(files), None
    mode,_=cnt.most_common(1)[0]
    kept=[fp for fp,s in sizes if s==mode]
    return kept, len(files)-len(kept), mode

def _embedder():
    try:
        m=torch.hub.load("facebookresearch/dinov2","dinov2_vits14",verbose=False).to(DEV).eval()
        from torchvision import transforms as T2
        pre=T2.Compose([T2.Resize(256),T2.CenterCrop(224),T2.ToTensor(),
                        T2.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        @torch.no_grad()
        def enc(paths):
            o=[]
            for i in range(0,len(paths),16):
                b=torch.stack([pre(Image.open(p).convert("RGB")) for p in paths[i:i+16]]).to(DEV)
                o.append(tF.normalize(m(b),dim=-1).cpu())
            return torch.cat(o)
        return enc,"dinov2"
    except Exception as e:
        print(f"[S0] DINOv2 unavailable ({type(e).__name__}); thumbnail fallback")
        def enc(paths):
            v=[]
            for p in paths:
                with Image.open(p) as im:
                    a=np.asarray(im.convert("L").resize((64,64)),np.float32).ravel()
                a=a-a.mean(); v.append(a/(np.linalg.norm(a)+1e-9))
            return torch.from_numpy(np.stack(v))
        return enc,"thumb"

def scene_audit(splits, role, enc, method):
    keys=[k for k in ("E1","E2","T","H") if splits.get(k)]
    emb={k:enc(splits[k]) for k in keys}
    worst=0.0; pair=None
    for a,b in itertools.combinations(keys,2):
        v=float((emb[a]@emb[b].T).max())
        if v>worst: worst,pair=v,f"{a}-{b}"
    print(f"[S0]   {role} worst cross-split scene-sim = {worst:.3f} ({pair}, {method})")
    if worst>=C.SCENE_SIM_HALT:
        HALT(f"{role}: cross-split near-duplicate {worst:.3f} ({pair}) >= {C.SCENE_SIM_HALT}. "
             f"Raise C.GUARD and re-run S0.")
    return {"method":method,"worst":worst,"pair":pair}

def s0_manifest():
    dv=discover_devices(); print(f"[S0] discovered {len(dv)} devices")
    by={}
    for d,v in dv.items(): by.setdefault(v["model"],[]).append(d)
    nAB=need_for("A")
    if C.PREF_PAIR_MODEL in by and sum(len(dv[d]["files"])>=nAB for d in by[C.PREF_PAIR_MODEL])>=2:
        pm=C.PREF_PAIR_MODEL
    else:
        pm=next((mo for mo,ds in sorted(by.items())
                 if sum(len(dv[d]["files"])>=nAB for d in ds)>=2), None)
    if pm is None: HALT(f"no same-model pair with >= {nAB} images")
    A,B=sorted([d for d in by[pm] if len(dv[d]["files"])>=nAB],
               key=lambda d:-len(dv[d]["files"]))[:2]
    picks={"A":A,"B":B}
    for role,pref in (("C",C.DEV_C_PREF),("D",C.DEV_D_PREF),("D2",C.DEV_D2_PREF)):
        cand=pref if pref in dv else None
        if cand is None:
            cand=next(d for d in sorted(dv) if dv[d]["model"]!=pm and d not in picks.values()
                      and len(dv[d]["files"])>=need_for(role))
        if dv[cand]["model"]==pm: HALT(f"{role}={cand} shares camera model with A/B — invalid null")
        picks[role]=cand
    print(f"[S0] picks: {picks} (A,B same model = {pm})")

    enc,method=_embedder()
    man={"config_sha":CFG_SHA,"picks":picks,"meas":MEAS,"pair_model":pm,
         "devices":{},"scene_audit":{}}
    hdr=["role","device","split","idx","path"]
    csvp=os.path.join(ROOT,"manifest","manifest.csv")
    if os.path.exists(csvp): os.remove(csvp)
    for role in ROLES:
        did=picks[role]; kept,dropped,size=uniform_filter(dv[did]["files"])
        if size is None: HALT(f"{did}: no readable images")
        if min(size)<MEAS: HALT(f"{did}: {size} < MEAS {MEAS}")
        need=need_for(role)
        print(f"[S0] {role}={did}: kept {len(kept)} (dropped {dropped}) size={size} need={need}")
        if len(kept)<need: HALT(f"{did}: {len(kept)} < {need}. Lower N_E*/GUARD or change device.")
        g=C.GUARD; i=0; sp={}
        for name,n in (("E1",n_e1(role)),("E2",n_e2(role)),("T",C.N_T),("H",C.N_H)):
            if n==0: continue
            sp[name]=kept[i:i+n]; i+=n+g
        seen=set()
        for k,v in sp.items():
            if seen&set(v): HALT(f"{role}: split overlap in {k}")
            seen|=set(v)
        man["scene_audit"][role]=scene_audit(sp, role, enc, method)
        man["devices"][role]={"id":did,"model":dv[did]["model"],"size":list(size),
                              "n":{k:len(v) for k,v in sp.items()},"guard":g}
        for k,v in sp.items():
            for j,fp in enumerate(v): append_row(csvp,hdr,[role,did,k,j,fp])
    json.dump(man, open(os.path.join(ROOT,"manifest","manifest.json"),"w"), indent=1)
    print("[S0] manifest written. PASS.")
    return man

if stage("S0"): MAN=s0_manifest()
else:
    MAN=json.load(open(os.path.join(ROOT,"manifest","manifest.json")))
    print("[S0] loaded manifest:", MAN["picks"])


In [ ]:
# %% S1 — FINGERPRINT BANK (E1 for injection, E2 for measurement) + quality gates
import csv as _csv
def read_manifest():
    M={}
    for r in _csv.DictReader(open(os.path.join(ROOT,"manifest","manifest.csv"))):
        M.setdefault((r["role"],r["split"]),[]).append(r["path"])
    return M

def K_path(role, which="E2", comp="K"): return os.path.join(ROOT,"fingerprints",f"{comp}_{role}_{which}.npy")

def s1_fingerprints():
    M=read_manifest(); bank={}
    for role in ROLES:
        for which in (("E1","E2") if has_E1(role) else ("E2",)):
            p=K_path(role,which)
            if os.path.exists(p):
                K=np.load(p); print(f"[S1] K_{role}_{which}: cached")
            else:
                t0=time.time(); K=estimate_K(M[(role,which)])
                np.save(p,K); print(f"[S1] K_{role}_{which}: n={len(M[(role,which)])} ({time.time()-t0:.0f}s)")
            if which=="E2":
                Kd,Kp,frac=defect_split(K)
                np.save(K_path(role,"E2","Kdef"),Kd); np.save(K_path(role,"E2","Kprnu"),Kp)
                print(f"[S1]   defect frac={frac:.5f}")
                bank[role]=K
    nAB=ncc0(bank["A"],bank["B"])
    print(f"[S1] NCC(K_A,K_B) same model, post-cleaning = {nAB:.5f}   <- kappa_model bound")
    # E1 vs E2 independence check (must be ~ the true fingerprint correlation, not 1.0)
    for r in ("A","B"):
        print(f"[S1] NCC(K_{r}_E1, K_{r}_E2) = {ncc0(np.load(K_path(r,'E1')), bank[r]):.4f} "
              f"(same device, disjoint image sets)")

    hdr=["role_img","role_K","idx","rho_mult","rho_add","pce0","pce_max"]
    csvp=os.path.join(ROOT,"csv","s1_positive_control.csv")
    done=done_keys(csvp,["role_img","role_K","idx"])
    for role in ROLES:
        for i,fp in enumerate(M[(role,"H")]):
            if all((role,rk,str(i)) in done for rk in ROLES): continue
            Z=load_lum_crop(fp); Wz=wavelet_residual(Z)
            for rk in ROLES:
                if (role,rk,str(i)) in done: continue
                rm,ra=rho_pair(Wz,Z,bank[rk])
                p0,pm,_=pce_from_surface(xcorr_surface(Wz,Z*bank[rk]))
                append_row(csvp,hdr,[role,rk,i,f"{rm:.6e}",f"{ra:.6e}",f"{p0:.3f}",f"{pm:.3f}"])
        print(f"[S1] positive control: {role} done")
    rows=list(_csv.DictReader(open(csvp)))
    same=np.array([float(r["rho_mult"]) for r in rows if r["role_img"]==r["role_K"]])
    diff=np.array([float(r["rho_mult"]) for r in rows if r["role_img"]!=r["role_K"]])
    lab=np.r_[np.ones(len(same)),np.zeros(len(diff))]; sc=np.r_[same,diff]
    o=np.argsort(sc); rk_=np.empty(len(sc)); rk_[o]=np.arange(len(sc))
    auc=float((rk_[lab==1].mean()-(lab.sum()-1)/2)/(len(sc)-lab.sum()))
    rho_A=float(np.median([float(r["rho_mult"]) for r in rows
                           if r["role_img"]=="A" and r["role_K"]=="A"]))
    rho_B=float(np.median([float(r["rho_mult"]) for r in rows
                           if r["role_img"]=="B" and r["role_K"]=="B"]))
    # same-model pair AUC — this, not the pooled figure, is what the design depends on
    ab=[r for r in rows if r["role_img"] in ("A","B") and r["role_K"] in ("A","B")]
    s2_=np.array([float(r["rho_mult"]) for r in ab if r["role_img"]==r["role_K"]])
    d2_=np.array([float(r["rho_mult"]) for r in ab if r["role_img"]!=r["role_K"]])
    l2=np.r_[np.ones(len(s2_)),np.zeros(len(d2_))]; c2=np.r_[s2_,d2_]
    o2=np.argsort(c2); r2=np.empty(len(c2)); r2[o2]=np.arange(len(c2))
    auc_pair=float((r2[l2==1].mean()-(l2.sum()-1)/2)/(len(c2)-l2.sum()))
    # variance decomposition of the fingerprint estimate (fractions of total)
    Ssame=ncc0(np.load(K_path("A","E1")), bank["A"])     # same device, disjoint images
    kappa=nAB                                            # shared camera-model component
    dev=max(Ssame-kappa,0.0); noise=max(1.0-Ssame,0.0)
    devfrac_contrast = dev/max(dev+noise,1e-9)           # what the paired S_A/S_B test sees
    print(f"[S1] AUC pooled(all roles) = {auc:.4f}   AUC same-model pair(A vs B) = {auc_pair:.4f}")
    print(f"[S1] rho_real: A={rho_A:.5f}  B={rho_B:.5f}")
    print(f"[S1] K-hat variance split: kappa_model={kappa:.3f}  device-unique={dev:.3f}  "
          f"estimation-noise={noise:.3f}")
    print(f"[S1] device share of the paired contrast = {devfrac_contrast:.3f} "
          f"(kappa cancels in S_A/S_B by construction)")
    fails=[]
    if auc<C.AUC_GATE: fails.append(f"pooled AUC {auc:.4f} < {C.AUC_GATE}")
    if auc_pair<C.AUC_PAIR_GATE: fails.append(f"same-model AUC {auc_pair:.4f} < {C.AUC_PAIR_GATE}")
    if kappa>C.KAPPA_CEIL: fails.append(f"kappa_model {kappa:.3f} > {C.KAPPA_CEIL} "
                                        f"(cleaning is not removing shared ISP/JPEG structure)")
    if devfrac_contrast<C.DEVFRAC_FLOOR: fails.append(
        f"device share of contrast {devfrac_contrast:.3f} < {C.DEVFRAC_FLOOR} "
        f"(raise N_E2, or move to a camera model with more instances)")
    if fails: HALT("K-quality gates FAILED:\n  - "+"\n  - ".join(fails))
    json.dump({"auc":auc,"auc_pair":auc_pair,"rho_real_A":rho_A,"rho_real_B":rho_B,"ncc_AB":nAB,
               "kappa":kappa,"dev_frac":dev,"noise_frac":noise,"devfrac_contrast":devfrac_contrast,
               "meas":MEAS,"config_sha":CFG_SHA},
              open(os.path.join(ROOT,"fingerprints","s1_gates.json"),"w"),indent=1)
    print("[S1] PASS. NOTE: lambda_min is NOT computed here — it comes from the empirical")
    print("     null in S7 (off-peak of the mean cross-correlation surface), which makes no")
    print("     pixel-independence assumption. See design section 14.1.")

if stage("S1"): s1_fingerprints()


In [ ]:
# %% S1b — EXACT SD-3.5 VAE ROUND-TRIP GATE (survey 5.4): measure survival, don't assume it
def s1b_vae_gate():
    outp=os.path.join(ROOT,"fingerprints","s1b_vae.json")
    if os.path.exists(outp):
        print("[S1b] cached:", json.load(open(outp))); return
    if DEV!="cuda": HALT("S1b needs a GPU (VAE forward pass)")
    from diffusers import AutoencoderKL
    vae=AutoencoderKL.from_pretrained(C.HF_MODEL, subfolder="vae", torch_dtype=torch.float32).to(DEV).eval()
    M=read_manifest()
    bank={r:np.load(K_path(r,"E2")) for r in ("A","B")}
    hdr=["role","idx","stage","rho_A","rho_B","pce0_own"]
    csvp=os.path.join(ROOT,"csv","s1b_vae_roundtrip.csv")
    done=done_keys(csvp,["role","idx","stage"])
    for role in ("A","B"):
        for i,fp in enumerate(M[(role,"H")]):
            if (role,str(i),"post") in done: continue
            rgb=load_rgb_crop(fp)
            Y=(0.299*rgb[...,0]+0.587*rgb[...,1]+0.114*rgb[...,2]).astype(np.float32)
            with torch.no_grad():
                x=torch.from_numpy(rgb/127.5-1.0).permute(2,0,1)[None].to(DEV)
                lat=vae.encode(x).latent_dist.mode()
                rec=vae.decode(lat).sample[0].permute(1,2,0).cpu().numpy()
            R=np.clip((rec+1.0)*127.5,0,255).astype(np.float32)
            Yr=(0.299*R[...,0]+0.587*R[...,1]+0.114*R[...,2]).astype(np.float32)
            for nm,img in (("pre",Y),("post",Yr)):
                W=wavelet_residual(img)
                ra=ncc0(W,img*bank["A"]); rb=ncc0(W,img*bank["B"])
                own=ra if role=="A" else rb
                p0,_,_=pce_from_surface(xcorr_surface(W,img*bank[role]))
                append_row(csvp,hdr,[role,i,nm,f"{ra:.6e}",f"{rb:.6e}",f"{p0:.2f}"])
        print(f"[S1b] {role} round-trip done")
    rows=list(_csv.DictReader(open(csvp)))
    def m(role,stg,col): return float(np.median([float(r[col]) for r in rows
                                                 if r["role"]==role and r["stage"]==stg]))
    res={}
    for stg in ("pre","post"):
        same=[float(r["rho_A"]) for r in rows if r["role"]=="A" and r["stage"]==stg]+ \
             [float(r["rho_B"]) for r in rows if r["role"]=="B" and r["stage"]==stg]
        cross=[float(r["rho_B"]) for r in rows if r["role"]=="A" and r["stage"]==stg]+ \
              [float(r["rho_A"]) for r in rows if r["role"]=="B" and r["stage"]==stg]
        s=np.array(same); d=np.array(cross)
        lab=np.r_[np.ones(len(s)),np.zeros(len(d))]; sc=np.r_[s,d]
        o=np.argsort(sc); rr=np.empty(len(sc)); rr[o]=np.arange(len(sc))
        res[stg]={"auc":float((rr[lab==1].mean()-(lab.sum()-1)/2)/(len(sc)-lab.sum())),
                  "rho_same_median":float(np.median(s))}
    res["retention"]=res["post"]["rho_same_median"]/max(res["pre"]["rho_same_median"],1e-12)
    res["config_sha"]=CFG_SHA
    json.dump(res, open(outp,"w"), indent=1)
    print(f"[S1b] pre  AUC={res['pre']['auc']:.4f} rho={res['pre']['rho_same_median']:.5f}")
    print(f"[S1b] post AUC={res['post']['auc']:.4f} rho={res['post']['rho_same_median']:.5f} "
          f"retention={res['retention']:.3f}")
    if res["post"]["auc"] < C.VAE_AUC_GATE:
        HALT(f"VAE round-trip gate FAILED: post-VAE device AUC {res['post']['auc']:.3f} < "
             f"{C.VAE_AUC_GATE}. The transformer never receives a usable fingerprint — "
             f"report this as the mechanism result and stop before training.")
    print("[S1b] PASS — fingerprint survives the exact SD-3.5 VAE.")

if stage("S1b"): s1b_vae_gate()


In [ ]:
# %% S1c — TRAINING-DATA MATERIALIZATION (all arms as PNG) + causal clean/swap + suppression gate
def train_dir(role, variant): return os.path.join(ROOT,"train_png",f"{role}_{variant}")

def s1c_materialize():
    M=read_manifest()
    K1={r:np.load(K_path(r,"E1")) for r in ("A","B")}     # E1: used ONLY to modify images
    K2={r:np.load(K_path(r,"E2")) for r in ROLES}         # E2: used ONLY to measure
    # --- suppression / injection efficacy on HELD-OUT reals (this is the gate) ---
    gp=os.path.join(ROOT,"fingerprints","s1c_suppression.json")
    if not os.path.exists(gp):
        raw_r=[]; cln_r=[]; swp_b=[]; raw_b=[]
        for fp in M[("A","H")]:
            rgb=load_rgb_crop(fp)
            for tag,img in (("raw",rgb),("clean",apply_clean(rgb,K1["A"])),
                            ("swap",apply_swap(rgb,K1["A"],K1["B"]))):
                Y=(0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]).astype(np.float32)
                W=wavelet_residual(Y)
                if tag=="raw": raw_r.append(ncc0(W,Y*K2["A"])); raw_b.append(ncc0(W,Y*K2["B"]))
                elif tag=="clean": cln_r.append(ncc0(W,Y*K2["A"]))
                else: swp_b.append(ncc0(W,Y*K2["B"]))
        sup={"rho_A_raw":float(np.median(raw_r)),"rho_A_clean":float(np.median(cln_r)),
             "rho_B_on_raw":float(np.median(raw_b)),"rho_B_on_swap":float(np.median(swp_b))}
        sup["suppression_factor"]=sup["rho_A_raw"]/max(abs(sup["rho_A_clean"]),1e-12)
        sup["injection_gain"]=sup["rho_B_on_swap"]/max(abs(sup["rho_B_on_raw"]),1e-12)
        sup["config_sha"]=CFG_SHA
        json.dump(sup, open(gp,"w"), indent=1)
    else: sup=json.load(open(gp))
    print(f"[S1c] rho(A_raw->K_A)={sup['rho_A_raw']:.5f}  rho(A_clean->K_A)={sup['rho_A_clean']:.5f}"
          f"  suppression={sup['suppression_factor']:.1f}x")
    print(f"[S1c] rho(raw->K_B)={sup['rho_B_on_raw']:.5f}  rho(swap->K_B)={sup['rho_B_on_swap']:.5f}"
          f"  injection gain={sup['injection_gain']:.1f}x")
    if sup["suppression_factor"] < C.SUPPRESS_GATE:
        HALT(f"suppression {sup['suppression_factor']:.1f}x < {C.SUPPRESS_GATE}x — the clean arm "
             f"would not be interpretable. Raise N_E1 or CLEAN_ALPHA.")
    # --- materialize every training set as PNG (fix #10: no JPEG-vs-PNG confound across arms) ---
    variants=set((a[1],a[2]) for a in all_arms() if a[1] is not None)
    for role,variant in sorted(variants):
        d=train_dir(role,variant); os.makedirs(d,exist_ok=True)
        files=M[(role,"T")]
        if len(glob.glob(os.path.join(d,"*.png")))>=len(files):
            print(f"[S1c] skip {role}_{variant} (materialized)"); continue
        for i,fp in enumerate(files):
            rgb=load_rgb_crop(fp)
            if variant=="clean": rgb=apply_clean(rgb,K1[role])
            elif variant=="swap": rgb=apply_swap(rgb,K1["A"],K1["B"])
            Image.fromarray(np.clip(rgb,0,255).astype(np.uint8)).save(
                os.path.join(d,f"{i:04d}.png"), compress_level=1)
        print(f"[S1c] materialized {role}_{variant}: {len(files)} PNG crops @ {MEAS}px")

if stage("S1c"): s1c_materialize()


In [ ]:
# %% S2 — LoRA TRAINING (arms from all_arms(); PNG training crops; resumable)
def adapter_dir(tag): return os.path.join(ROOT,"adapters",tag)

def train_arm(tag):
    _,role,variant,seed,cap = arm_of(tag)
    if role is None: return
    out=adapter_dir(tag)
    if os.path.exists(os.path.join(out,"pytorch_lora_weights.safetensors")):
        print(f"[S2] skip existing {tag}"); return
    if DEV!="cuda": HALT("S2 requires a GPU")
    import random
    from diffusers import StableDiffusion3Pipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    pipe=StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL, torch_dtype=torch.bfloat16).to(DEV)
    with torch.no_grad():
        pe,_,pp,_=pipe.encode_prompt(prompt=C.CAPTION,prompt_2=C.CAPTION,prompt_3=C.CAPTION,
                                     device=DEV,num_images_per_prompt=1,
                                     do_classifier_free_guidance=False)
    pipe.text_encoder=pipe.text_encoder_2=pipe.text_encoder_3=None; torch.cuda.empty_cache()
    vae,tr=pipe.vae,pipe.transformer
    vae.requires_grad_(False); tr.requires_grad_(False)
    rank=C.LORA_RANK if cap=="r16" else C.CEIL_RANK
    tgts=list(C.LORA_TARGETS if cap=="r16" else C.CEIL_TARGETS)
    tr.add_adapter(LoraConfig(r=rank,lora_alpha=rank,init_lora_weights="gaussian",target_modules=tgts))
    tr.enable_gradient_checkpointing()
    params=[p for p in tr.parameters() if p.requires_grad]
    steps=C.STEPS if cap=="r16" else C.CEIL_STEPS
    opt=torch.optim.AdamW(params,lr=C.LR,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=steps)
    files=sorted(glob.glob(os.path.join(train_dir(role,variant),"*.png")))
    if not files: HALT(f"no materialized training PNGs for {role}_{variant} — run S1c")
    cache=torch.stack([torch.from_numpy(np.asarray(Image.open(f).convert("RGB"),np.float32)/127.5-1.0)
                       .permute(2,0,1) for f in files])
    print(f"[S2] {tag}: {len(files)} imgs @ {MEAS}px rank{rank} {steps} steps")
    sf,sh=vae.config.scaling_factor, vae.config.shift_factor
    g=torch.Generator().manual_seed(seed); losses=[]
    for st in range(steps):
        idx=torch.randint(0,cache.shape[0],(C.BATCH,),generator=g)
        px=cache[idx].to(DEV,torch.bfloat16)
        with torch.no_grad():
            lat=vae.encode(px).latent_dist.sample(); lat=(lat-sh)*sf
        noise=torch.randn_like(lat)
        u=torch.sigmoid(torch.randn(lat.shape[0],device=DEV)); t=u.view(-1,1,1,1)
        pred=tr(hidden_states=(1.0-t)*lat+t*noise, timestep=u*1000,
                encoder_hidden_states=pe.repeat(lat.shape[0],1,1),
                pooled_projections=pp.repeat(lat.shape[0],1), return_dict=False)[0]
        loss=tF.mse_loss(pred.float(),(noise-lat).float())/C.GRAD_ACC
        loss.backward()
        if (st+1)%C.GRAD_ACC==0:
            torch.nn.utils.clip_grad_norm_(params,1.0)
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()*C.GRAD_ACC)
        if (st+1)%200==0: print(f"    {st+1}/{steps} loss {np.mean(losses[-200:]):.4f}")
    os.makedirs(out,exist_ok=True)
    StableDiffusion3Pipeline.save_lora_weights(out, transformer_lora_layers=get_peft_model_state_dict(tr))
    json.dump({"loss_tail":float(np.mean(losses[-100:])),"meas":MEAS,"rank":rank,"steps":steps,
               "role":role,"variant":variant,"seed":seed,"config_sha":CFG_SHA},
              open(os.path.join(out,"train_meta.json"),"w"))
    del pipe,tr,vae; torch.cuda.empty_cache()
    print(f"[S2] saved {tag}")

if stage("S2"):
    for tag,role,_,_,_ in all_arms():
        if role is not None: train_arm(tag)


In [ ]:
# %% S3 — GENERATION with a PAIRED seed bank (identical (prompt, eps) for every adapter)
DIVERSE_PROMPTS = ["a photograph of a street", "a photograph of a room interior",
                   "a photograph of trees", "a photograph of a building facade",
                   "a photograph of a table with objects"]

def prompt_seed_bank():
    """Pre-registered, identical across adapters -> paired tests (survey 5.7)."""
    bank=[]
    for j in range(C.G_PER_ADAPTER):
        p = C.CAPTION if C.PROMPT_BANK=="uniform" else DIVERSE_PROMPTS[j%len(DIVERSE_PROMPTS)]
        bank.append((p, C.GEN_SEED_BASE+j))
    return bank

def gen_dir(tag): return os.path.join(ROOT,"gens",tag)

def generate(tag):
    outd=gen_dir(tag); os.makedirs(outd,exist_ok=True)
    bank=prompt_seed_bank()
    have=len(glob.glob(os.path.join(outd,"*.png")))
    if have>=len(bank): print(f"[S3] skip {tag} ({have}/{len(bank)})"); return
    if DEV!="cuda": HALT("S3 requires a GPU")
    from diffusers import StableDiffusion3Pipeline
    pipe=StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL,torch_dtype=torch.bfloat16).to(DEV)
    if tag!="base": pipe.load_lora_weights(adapter_dir(tag))
    assert getattr(pipe,"image_encoder",None) is None, "txt2img only — no image conditioning"
    i=have
    while i<len(bank):
        chunk=bank[i:i+C.GEN_BATCH]
        gs=[torch.Generator(device=DEV).manual_seed(s) for _,s in chunk]
        with torch.no_grad():
            imgs=pipe(prompt=[p for p,_ in chunk],num_inference_steps=C.GEN_STEPS,
                      guidance_scale=C.CFG_SCALE,height=MEAS,width=MEAS,generator=gs).images
        for j,im in enumerate(imgs): im.save(os.path.join(outd,f"{i+j:05d}.png"))
        i+=len(chunk)
        if i%100<C.GEN_BATCH: print(f"[S3] {tag}: {i}/{len(bank)}")
    del pipe; torch.cuda.empty_cache()

if stage("S3"):
    for tag in all_tags(): generate(tag)


In [ ]:
# %% S4 — COPY ENSEMBLE (DINOv2 full + quadrant crops, pHash, residual PEAKEDNESS)
def _open_rgb(p):
    with Image.open(p) as im: return im.convert("RGB").copy()

def s4_copy_ensemble():
    import imagehash
    dino=torch.hub.load("facebookresearch/dinov2","dinov2_vits14",verbose=False).to(DEV).eval()
    from torchvision import transforms as T2
    pre=T2.Compose([T2.Resize(256),T2.CenterCrop(224),T2.ToTensor(),
                    T2.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    @torch.no_grad()
    def enc_imgs(imgs):
        o=[]
        for i in range(0,len(imgs),16):
            o.append(tF.normalize(dino(torch.stack([pre(im) for im in imgs[i:i+16]]).to(DEV)),
                                  dim=-1).cpu())
        return torch.cat(o)
    def enc_paths(paths):
        o=[]
        for i in range(0,len(paths),16):          # streamed: never holds >16 full images
            o.append(enc_imgs([_open_rgb(p) for p in paths[i:i+16]]))
        return torch.cat(o)
    def quads(im):
        w,h=im.size
        return [im.crop((0,0,w//2,h//2)),im.crop((w//2,0,w,h//2)),
                im.crop((0,h//2,w//2,h)),im.crop((w//2,h//2,w,h))]
    M=read_manifest()
    hdr=["tag","gen_idx","dino_sim","crop_sim","phash_min","resid_z","copy_flag"]
    csvp=os.path.join(ROOT,"csv","s4_copy.csv"); done=done_keys(csvp,["tag","gen_idx"])
    ref={}
    for tag in all_tags():
        _,role,variant,_,_=arm_of(tag)
        rk = role if role is not None else "A"
        rv = variant if variant is not None else "raw"
        key=(rk,rv)
        if key not in ref:
            tp=sorted(glob.glob(os.path.join(train_dir(rk,rv),"*.png")))   # what it trained on
            ep=M[(rk,"E2")]                                                # never trained on
            if not tp: HALT(f"no materialized training PNGs for {rk}_{rv} — run S1c")
            phs=[]
            for p in tp+ep:
                im=_open_rgb(p); phs.append(imagehash.phash(im)); del im
            # residual reference: TRAIN split only (you can only memorize what you trained on),
            # float16 (peakedness needs no precision) -> ~2 MB/image instead of 4
            resid=np.stack([wavelet_residual(load_lum_crop(p,use_cache=False)).astype(np.float16)
                            for p in tp])
            ref[key]={"emb":enc_paths(tp+ep),"phash":phs,"resid":resid,"n_train":len(tp)}
            print(f"[S4] reference {rk}_{rv}: {len(tp)} train + {len(ep)} held-out, "
                  f"resid bank {resid.nbytes/1e6:.0f} MB")
        R=ref[key]
        paths=sorted(glob.glob(os.path.join(gen_dir(tag),"*.png")))
        todo=[(i,p) for i,p in enumerate(paths) if (tag,str(i)) not in done]
        for i0 in range(0,len(todo),32):
            ch=todo[i0:i0+32]
            gims=[_open_rgb(p) for _,p in ch]
            dsim=(enc_imgs(gims)@R["emb"].T).max(dim=1).values.numpy()
            csim=(enc_imgs([q for im in gims for q in quads(im)])@R["emb"].T)\
                 .max(dim=1).values.numpy().reshape(len(ch),4).max(axis=1)
            for k,(gi,p) in enumerate(ch):
                ph=min(int(imagehash.phash(gims[k])-h) for h in R["phash"])
                W=wavelet_residual(load_lum_crop(p,use_cache=False))
                cc=np.array([ncc0(W,r.astype(np.float32)) for r in R["resid"]])
                med=float(np.median(cc)); mad=float(np.median(np.abs(cc-med)))/0.6745+1e-12
                rz=float((cc.max()-med)/mad)   # PEAKEDNESS: a shared fingerprint lifts ALL
                                               # correlations; a memorized copy lifts exactly one.
                                               # Raw max (as the survey proposed) would flag the
                                               # leakage signal itself as copying.
                flag=int(dsim[k]>=C.COPY_DINO or csim[k]>=C.COPY_CROP or
                         ph<=C.COPY_PHASH or rz>=C.COPY_RESID_Z)
                append_row(csvp,hdr,[tag,gi,f"{dsim[k]:.4f}",f"{csim[k]:.4f}",ph,f"{rz:.2f}",flag])
            del gims
        print(f"[S4] {tag}: {len(todo)} audited")
if stage("S4"): s4_copy_ensemble()


In [ ]:
# %% S5 — MEASUREMENT + cross-correlation surface accumulation (empirical null, free)
def s5_measure():
    K2={r:np.load(K_path(r,"E2")) for r in ROLES}
    KD={r:np.load(K_path(r,"E2","Kdef")) for r in ROLES}
    KP={r:np.load(K_path(r,"E2","Kprnu")) for r in ROLES}
    hdr=["tag","gen_idx","K","rho_mult","rho_add","pce0","pce_max","sh_r","sh_c","rho_def","rho_prnu"]
    csvp=os.path.join(ROOT,"csv","s5_measure.csv"); done=done_keys(csvp,["tag","gen_idx","K"])
    for tag in all_tags():
        paths=sorted(glob.glob(os.path.join(gen_dir(tag),"*.png")))
        surf_p=os.path.join(ROOT,"surfaces",f"{tag}__KA.npy")
        need_surf = not os.path.exists(surf_p)
        acc=np.zeros((MEAS,MEAS),np.float32) if need_surf else None
        t0=time.time(); n=0
        for gi,fp in enumerate(paths):
            if all((tag,str(gi),rk) in done for rk in ROLES) and not need_surf: continue
            Z=load_lum_crop(fp,use_cache=False); Wz=wavelet_residual(Z)
            if need_surf: acc += xcorr_surface(Wz, Z*K2["A"])
            for rk in ROLES:
                if (tag,str(gi),rk) in done: continue
                rm,ra=rho_pair(Wz,Z,K2[rk])
                Cs=xcorr_surface(Wz,Z*K2[rk]); p0,pm,sh=pce_from_surface(Cs)
                append_row(csvp,hdr,[tag,gi,rk,f"{rm:.6e}",f"{ra:.6e}",f"{p0:.3f}",f"{pm:.3f}",
                                     sh[0],sh[1],f"{ncc0(Wz,Z*KD[rk]):.6e}",f"{ncc0(Wz,Z*KP[rk]):.6e}"])
            n+=1
        if need_surf and paths:
            np.save(surf_p, (acc/len(paths)).astype(np.float32))
        print(f"[S5] {tag}: {n} gens ({time.time()-t0:.0f}s){' +surface' if need_surf else ''}")
if stage("S5"): s5_measure()


In [ ]:
# %% S6 — INTEGRITY AUDIT (halt before any statistic or figure)
def s6_audit():
    import pandas as pd
    df=pd.read_csv(os.path.join(ROOT,"csv","s5_measure.csv"))
    cp=pd.read_csv(os.path.join(ROOT,"csv","s4_copy.csv"))
    nK=len(ROLES); probs=[]
    if df.isna().any().any(): probs.append("NaN in s5_measure")
    if cp.isna().any().any(): probs.append("NaN in s4_copy")
    for tag,g in df.groupby("tag"):
        if len(g)!=C.G_PER_ADAPTER*nK: probs.append(f"row parity {tag}: {len(g)} != {C.G_PER_ADAPTER*nK}")
        if float(g["rho_mult"].std())==0.0: probs.append(f"zero variance {tag}")
    miss=set(all_tags())-set(df["tag"])
    if miss: probs.append(f"missing tags in S5: {sorted(miss)}")
    if set(cp["tag"])!=set(df["tag"]): probs.append("copy-audit / measure tag mismatch")
    for t in all_tags():
        if not os.path.exists(os.path.join(ROOT,"surfaces",f"{t}__KA.npy")):
            probs.append(f"missing null surface for {t}")
    for f,nm in ((os.path.join(ROOT,"fingerprints","s1_gates.json"),"S1"),
                 (os.path.join(ROOT,"fingerprints","s1b_vae.json"),"S1b"),
                 (os.path.join(ROOT,"fingerprints","s1c_suppression.json"),"S1c")):
        if not os.path.exists(f): probs.append(f"{nm} gate artefact missing")
        elif json.load(open(f)).get("config_sha")!=CFG_SHA:
            probs.append(f"{nm} protocol SHA drift")
    # paired-seed integrity: every tag must have the same number of gens
    counts={t:len(glob.glob(os.path.join(gen_dir(t),"*.png"))) for t in all_tags()}
    if len(set(counts.values()))!=1: probs.append(f"unpaired generation counts: {counts}")
    if probs: HALT("integrity audit FAILED:\n  "+"\n  ".join(probs))
    print(f"[S6] PASS — {len(df)} rows, {df['tag'].nunique()} tags, paired counts, SHA consistent.")
if stage("S6"): s6_audit()


In [ ]:
# %% S7 — STATISTICS: empirical lambda_min, IUT primary, Holm secondaries, C0-C4 verdict
def holm(pvals, alpha):
    """Holm-Bonferroni. Returns dict name -> (p, adj_threshold, reject)."""
    items=sorted(pvals.items(), key=lambda kv: kv[1]); m=len(items); out={}; still=True
    for i,(k,p) in enumerate(items):
        thr=alpha/(m-i)
        rej = still and (p<=thr)
        if not rej: still=False
        out[k]=(p,thr,bool(rej))
    return out

def energy_distance(x,y):
    """Sign-agnostic two-sample statistic (survey 5.10)."""
    x=np.sort(x); y=np.sort(y)
    def _m(a,b):
        return np.abs(a[:,None]-b[None,:]).mean() if len(a)*len(b)<4_000_000 else \
               float(np.mean([np.abs(v-b).mean() for v in a]))
    return 2*_m(x,y)-_m(x,x)-_m(y,y)

def s7_stats():
    import pandas as pd
    from scipy import stats as sps
    rng=np.random.default_rng(0)
    df=pd.read_csv(os.path.join(ROOT,"csv","s5_measure.csv"))
    cp=pd.read_csv(os.path.join(ROOT,"csv","s4_copy.csv"))
    df=df.merge(cp[["tag","gen_idx","copy_flag","dino_sim","resid_z"]],on=["tag","gen_idx"],how="left")
    g1=json.load(open(os.path.join(ROOT,"fingerprints","s1_gates.json")))
    vae=json.load(open(os.path.join(ROOT,"fingerprints","s1b_vae.json")))
    sup=json.load(open(os.path.join(ROOT,"fingerprints","s1c_suppression.json")))
    rho_A, rho_B = g1["rho_real_A"], g1["rho_real_B"]

    def sel(tag,K,col="rho_mult",excl=False):
        m=(df.tag==tag)&(df.K==K)
        if excl: m&=(df.copy_flag==0)
        return df.loc[m,col].to_numpy(float)
    def cat(tags,K,col="rho_mult",excl=False):
        return np.concatenate([sel(t,K,col,excl) for t in tags])
    RAW_A=[f"A_raw_s{s}_r16" for s in C.TRAIN_SEEDS]
    RAW_B=[f"B_raw_s{s}_r16" for s in C.TRAIN_SEEDS]
    RAW_D=[f"D_raw_s{s}_r16" for s in C.TRAIN_SEEDS]

    def perm_p(x,y,two_sided=False):
        obs=x.mean()-y.mean(); z=np.r_[x,y]; n=len(x); c=0
        for _ in range(C.N_PERM):
            rng.shuffle(z); d=z[:n].mean()-z[n:].mean()
            c += (abs(d)>=abs(obs)) if two_sided else (d>=obs)
        return (c+1)/(C.N_PERM+1), float(obs)
    def perm_p_paired(d, two_sided=False):
        """Sign-flip test for a within-image paired contrast."""
        obs=d.mean(); c=0
        for _ in range(C.N_PERM):
            s=rng.choice((-1.0,1.0),size=len(d)); v=(d*s).mean()
            c += (abs(v)>=abs(obs)) if two_sided else (v>=obs)
        return (c+1)/(C.N_PERM+1), float(obs)
    def bca(x):
        r=sps.bootstrap((x,),np.mean,n_resamples=C.N_BOOT,method="BCa",
                        confidence_level=1-2*C.ALPHA,random_state=0)
        return float(r.confidence_interval.low), float(r.confidence_interval.high)

    # ---------- 1. EMPIRICAL null and lambda_min (supersedes the analytic 1/sqrt(mG)) ----------
    surf=np.mean([np.load(os.path.join(ROOT,"surfaces",f"{t}__KA.npy")) for t in RAW_A],axis=0)
    off=surf[offpeak_mask(surf.shape)]
    se_emp=float(off.std())
    z=sps.norm.ppf(1-C.ALPHA)+sps.norm.ppf(C.POWER)
    lam_min=float(z*se_emp/rho_A)
    se_analytic=1.0/(MEAS*math.sqrt(C.G_PER_ADAPTER))
    print(f"[S7] empirical SE(rho_bar) = {se_emp:.3e}   analytic 1/sqrt(mG) = {se_analytic:.3e}"
          f"   inflation = {se_emp/se_analytic:.2f}x")
    print(f"[S7] lambda_min (alpha={C.ALPHA}, power={C.POWER}) = {lam_min:.4%}")
    if lam_min>C.LAMBDA_MIN_CEIL:
        print(f"[S7] WARNING: lambda_min above the {C.LAMBDA_MIN_CEIL:.0%} ceiling — a null is weak.")

    # ---------- 2. PRIMARY: intersection-union test, no alpha adjustment ----------
    SA=cat(RAW_A,"A")-cat(RAW_A,"B")           # device A adapters: K_A vs K_B, same image
    SB=cat(RAW_B,"B")-cat(RAW_B,"A")           # symmetric replication
    pA,dA=perm_p_paired(SA); pB,dB=perm_p_paired(SB)
    loA,hiA=bca(SA); loB,hiB=bca(SB)
    PRIMARY = (pA<C.ALPHA) and (pB<C.ALPHA)    # IUT: both must pass at alpha; level is exact
    per_seed=[]
    for s in C.TRAIN_SEEDS:
        a=sel(f"A_raw_s{s}_r16","A")-sel(f"A_raw_s{s}_r16","B")
        b=sel(f"B_raw_s{s}_r16","B")-sel(f"B_raw_s{s}_r16","A")
        per_seed.append((perm_p_paired(a)[0], perm_p_paired(b)[0]))
        print(f"[S7] seed{s}: S_A p={per_seed[-1][0]:.4g}  S_B p={per_seed[-1][1]:.4g}")
    seed_ok = sum(pa<C.ALPHA and pb<C.ALPHA for pa,pb in per_seed) >= 2
    PRIMARY = PRIMARY and seed_ok
    print(f"[S7] PRIMARY IUT: S_A p={pA:.4g} (d={dA:.3e}) & S_B p={pB:.4g} (d={dB:.3e}) "
          f"-> {'PASS' if PRIMARY else 'fail'}")

    # ---------- 3. SECONDARY family (Holm at ALPHA_SEC), evaluated only if PRIMARY passes ----
    XA=cat(RAW_A,"A"); XD=cat(RAW_D,"A")
    sec={}
    sec["existence_vs_D"]   = perm_p(XA,XD)[0]
    sec["existence_vs_D2"]  = perm_p(XA,sel("D2_raw_s0_r16","A"))[0]
    sec["base_contamination"]=perm_p(sel("base","A"),XD)[0]
    sec["copy_excluded"]    = perm_p_paired(cat(RAW_A,"A",excl=True)-cat(RAW_A,"B",excl=True))[0]
    sec["causal_clean"]     = perm_p(XA,sel("A_clean_s0_r16","A"))[0]        # raw > clean
    sec["causal_swap_to_B"] = perm_p(sel("A_swap_s0_r16","B"),cat(RAW_A,"B"))[0]  # swap -> K_B
    sec["capacity_ceiling"] = perm_p(sel("A_raw_s0_ceil","A"),sel("D_raw_s0_ceil","A"))[0]
    sec["two_sided_any_dep"]= perm_p(XA,XD,two_sided=True)[0]
    ed_obs=energy_distance(XA,XD); zz=np.r_[XA,XD]; nA=len(XA); c=0
    for _ in range(1000):
        rng.shuffle(zz); c += energy_distance(zz[:nA],zz[nA:])>=ed_obs
    sec["energy_distance"]=(c+1)/1001
    holm_res = holm(sec, C.ALPHA_SEC) if PRIMARY else {k:(v,float("nan"),False) for k,v in sec.items()}
    for k,(p,thr,rej) in sorted(holm_res.items(), key=lambda kv: kv[1][0]):
        print(f"[S7]   {k:22s} p={p:.4g} thr={thr:.4g} {'REJECT H0' if rej else '-'}")

    # ---------- 4. C-ladder verdict ----------
    causal_ok = PRIMARY and holm_res["causal_clean"][2] and holm_res["causal_swap_to_B"][2]
    distrib_ok= PRIMARY and holm_res["copy_excluded"][2]
    if PRIMARY and causal_ok and distrib_ok: rung="C4"
    elif PRIMARY and causal_ok:              rung="C3"
    elif PRIMARY:                            rung="C2"
    elif holm_res["existence_vs_D"][0]<C.ALPHA_SEC: rung="C1"
    else:                                    rung="C0"
    meaning={"C0":f"power-bounded null: |lambda| < {lam_min:.3%} at alpha={C.ALPHA}, power={C.POWER}",
             "C1":"acquisition-pipeline dependence; device specificity NOT established",
             "C2":"passive device-level leakage (same-model devices separable)",
             "C3":"causal sensor-fingerprint leakage (vanishes on clean, follows the swap)",
             "C4":"causal + distributional (survives copy exclusion)"}[rung]
    lam_hat=float(XA.mean()/rho_A)
    out={"config_sha":CFG_SHA,"verdict":{"rung":rung,"meaning":meaning},
         "power":{"se_empirical":se_emp,"se_analytic":se_analytic,
                  "inflation":se_emp/se_analytic,"lambda_min":lam_min,"lambda_hat":lam_hat},
         "primary_IUT":{"p_SA":pA,"p_SB":pB,"d_SA":dA,"d_SB":dB,"ci_SA":[loA,hiA],
                        "ci_SB":[loB,hiB],"per_seed":per_seed,"pass":bool(PRIMARY)},
         "secondary_holm":{k:{"p":v[0],"thr":v[1],"reject":v[2]} for k,v in holm_res.items()},
         "gates":{"S1":g1,"S1b":vae,"S1c":sup},
         "components":{"rho_def":float(cat(RAW_A,"A","rho_def").mean()),
                       "rho_prnu":float(cat(RAW_A,"A","rho_prnu").mean())},
         "template":{"rho_add":float(cat(RAW_A,"A","rho_add").mean()),"rho_mult":float(XA.mean())},
         "n_copy_flagged":{t:int(df[(df.tag==t)&(df.K=="A")]["copy_flag"].sum()) for t in RAW_A}}
    json.dump(out,open(os.path.join(ROOT,"E_INV_P0_results.json"),"w"),indent=1)
    print(f"\n[S7] ================ VERDICT {rung} ================")
    print(f"[S7] {meaning}")
    print(f"[S7] lambda_hat={lam_hat:.3%}  lambda_min={lam_min:.3%}")

    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    F=os.path.join(ROOT,"figs")
    fig,ax=plt.subplots(figsize=(7,4))
    for lab,x in (("S_A (A-adapters: K_A-K_B)",SA),("S_B (B-adapters: K_B-K_A)",SB),
                  ("A-gens->K_A",XA),("D-gens->K_A (null)",XD)):
        ax.hist(x,bins=80,histtype="step",density=True,label=lab)
    ax.axvline(0,color="k",lw=0.5); ax.legend(fontsize=7); ax.set_xlabel("statistic")
    ax.set_title("primary contrasts vs null")
    fig.savefig(os.path.join(F,"fig1_primary.png"),dpi=160,bbox_inches="tight")
    fig,ax=plt.subplots(figsize=(6.5,4))
    arms=["A_raw","A_clean_s0_r16","A_swap_s0_r16","B_raw","D_raw","base"]
    vals=[cat(RAW_A,"A"),sel("A_clean_s0_r16","A"),sel("A_swap_s0_r16","B"),
          cat(RAW_B,"A"),XD,sel("base","A")]
    ax.bar(range(len(arms)),[v.mean() for v in vals],
           yerr=[v.std()/math.sqrt(len(v)) for v in vals])
    ax.set_xticks(range(len(arms))); ax.set_xticklabels(arms,rotation=30,ha="right",fontsize=7)
    ax.axhline(0,color="k",lw=0.5); ax.set_ylabel("mean rho (swap arm scored vs K_B)")
    ax.set_title("causal arms")
    fig.savefig(os.path.join(F,"fig2_causal_arms.png"),dpi=160,bbox_inches="tight")
    fig,ax=plt.subplots(figsize=(6,4))
    a=df[df.tag.isin(RAW_A)&(df.K=="A")]
    ax.scatter(a["resid_z"],a["rho_mult"],s=4,alpha=0.4)
    ax.axvline(C.COPY_RESID_Z,ls="--"); ax.set_xlabel("residual peakedness z (top1 vs median)")
    ax.set_ylabel("rho_mult vs K_A"); ax.set_title("memorization vs distributional")
    fig.savefig(os.path.join(F,"fig3_copy_scatter.png"),dpi=160,bbox_inches="tight")
    print("[S7] figures ->",F)
if stage("S7"): s7_stats()


## Multiplicity plan (pre-registered)

- **Primary** — one test, no correction: intersection–union of S_A>0 and S_B>0 at α=0.01
  one-sided. An IUT is exact at level α for the intersection null, so combining the two
  contrasts costs nothing. Additionally required: ≥2/3 training seeds pass both individually.
- **Secondary family** — 9 tests, Holm at α=0.05, **evaluated only if the primary passes**
  (hierarchical gating). Reported with p, Holm threshold, and decision.
- Everything else (component ordering, template comparison, λ̂) is **descriptive**, reported
  without inference.

## Sessions and budget

| # | STAGES | Wall-clock | Gate |
|---|---|---|---|
| 1 | `("S0",)` `("S1",)` | ~10 min / ~2 h | real-image device AUC ≥ 0.99 |
| 2 | `("S1b",)` | ~20 min GPU | post-VAE device AUC ≥ 0.75 — **kills the project cheaply if it fails** |
| 3 | `("S1c",)` | ~30 min | suppression ≥ 5×; PNG training sets materialized for all arms |
| 4–6 | `("S2",)` | ~20–26 A100-h (14 adapters, resumable) | loss tails |
| 7 | `("S3",)` | ~6 h (7500 gens, paired seeds) | equal counts per tag |
| 8 | `("S4","S5")` | ~4–6 h | CSV parity + null surfaces |
| 9 | `("S6","S7")` | minutes | **C0–C4 verdict** |

## Claim wording (do not drift)

Null result: *"A null under this maximal-sensitivity fixed-coordinate protocol is strong evidence
against zero-lag pixel-locked leakage for this architecture and adaptation method"* — **not**
"leakage does not occur". Positive result: passive, natural, device-level, txt2img-only,
same-model-discriminating. The swap arm is a **pipeline positive control**, never a novelty claim.

## Out of scope until rung ≥ C2

Certified FAR detector (patent gate first), dose–response over N and LoRA rank, random-crop
realism arm, additional device pairs (Kodak_M1063 has 5 instances, 438–571 images each — the
richest replication target in Dresden), smartphone replication (needs VISION/FODB, not Dresden),
2×2 architecture×density disentangling.